# ВАЖНО: Проверить флаг `run_blast_filter`

In [1]:
import config
import os

In [2]:
repbase_update_dir = config.DIR_REPBASE_RAW

In [3]:
run_blast_filter = True

## Считаем общее количество последовательностей RepBase

In [4]:
def count_fasta_sequences(root_dir):
    total_sequences = 0

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith("rep.ref"):
                file_path = os.path.join(dirpath, filename)

                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    for line in f:
                        if line.startswith(">"):
                            total_sequences += 1

    return total_sequences


total = count_fasta_sequences(repbase_update_dir)
print(f"Общее количество последовательностей: {total}")

Общее количество последовательностей: 119777


In [5]:
if run_blast_filter:
    with open(f"{config.DIR_REPBASE_PREPROCESSED}/remove_ids00.txt") as f:
        remove_ids = [line.strip() for line in f if line.strip()]

    print(remove_ids)

['ABR1B', 'ACASINE2b', 'ACASINE2c', 'ACASINE2d', 'AFROA1B', 'AFROSINE-1_LA', 'AFROSINE-2_LA', 'AFROSINE1B', 'AFROSINE2', 'AFROSINE3', 'ALPINE2', 'ALTR2B1_Vpa', 'ALTR2B1b_Vpa', 'ALTR2B2_SSc', 'ALTR2B3_SSc', 'ALTR2B_BT', 'ALTR2B_SSc', 'ALTR2C_BT', 'ALTR2_Ttr', 'ALTR2_Vpa', 'ALTR2b_Vpa', 'ANGELA1_TM_I', 'ANGELA6_TM_I', 'ANGELA6_TM_LTR', 'ATCOPIA31_I', 'ATCOPIA32_I', 'ATCOPIA38B_I', 'ATCOPIA38_I', 'ATCOPIA40_I', 'ATCOPIA40_LTR', 'ATCOPIA64LTR', 'ATCOPIA64_I', 'ATCOPIA65LTR', 'ATCOPIA69B_I', 'ATCOPIA69C-I', 'ATCOPIA69_I', 'ATCOPIA72_I', 'ATCOPIA73_I', 'ATCOPIA79LTR', 'ATCOPIA86_I', 'ATCOPIA8BLTR', 'ATDNA2T9C', 'ATGP3I', 'ATGP6I', 'ATGP7I', 'ATHILA-6_SBi-LTR', 'ATHILA4D_LTR', 'ATHILA4_LTR', 'ATHILA6C_LTR', 'AVMAR1A', 'Academ-11_ChOp', 'Academ-4_CorFlu', 'Academ-5_SiCo', 'Academ-6_CGi', 'Academ-8_MyCo', 'Academ-91_ChOp', 'AcademH-1_PTrit', 'AcademH-N2_PGr', 'AcademH-N2_PHor', 'AcademH-N3_PGr', 'AcademH-N4_PHor', 'AcademHP-1_CVi', 'AciJub-1.15', 'AciJub-1.17', 'AciJub-1.2', 'AciJub-1.221', 'Ac

In [6]:
def collect_fasta_headers(root_dir):
    headers = []

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith("rep.ref"):
                file_path = os.path.join(dirpath, filename)

                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    for line in f:
                        if line.startswith(">"):
                            headers.append(line.strip()[1:])

    return headers


headers = collect_fasta_headers(repbase_update_dir)

print(f"Всего заголовков: {len(headers)}")
print("\nПримеры заголовков:")
for h in headers[:10]:
    print(h)

Всего заголовков: 119777

Примеры заголовков:
MARINER62_CB	Mariner/Tc1	Caenorhabditis briggsae
GYPSY1-LTR_CB	Gypsy	Caenorhabditis briggsae
DNA2-13_CB	DNA transposon	Caenorhabditis briggsae
MINISAT6_CB	MSAT	Caenorhabditis briggsae
MINISAT4_CB	MSAT	Caenorhabditis briggsae
MARINER15_CB	Mariner/Tc1	Caenorhabditis briggsae
MARINER56_CB	Mariner/Tc1	Caenorhabditis briggsae
HELITRON11_CB	Helitron	Caenorhabditis briggsae
Polinton-1_CB	Polinton	Caenorhabditis briggsae
MARINER28_CB	Mariner/Tc1	Caenorhabditis briggsae


In [7]:
headers_with_tabs = [h for h in headers if "\t" in h]

print(f"Заголовков с табами: {len(headers_with_tabs)} / {len(headers)}")

from collections import Counter

split_lengths = Counter(len(h.split("\t")) for h in headers)

print("\nРаспределение количества колонок после split по табу:")
for k, v in sorted(split_lengths.items()):
    print(f"{k} колонок: {v} строк")

print("\nПримеры заголовков с табами:")
for h in headers_with_tabs[:5]:
    print(h.split("\t"))

Заголовков с табами: 119777 / 119777

Распределение количества колонок после split по табу:
3 колонок: 119777 строк

Примеры заголовков с табами:
['MARINER62_CB', 'Mariner/Tc1', 'Caenorhabditis briggsae']
['GYPSY1-LTR_CB', 'Gypsy', 'Caenorhabditis briggsae']
['DNA2-13_CB', 'DNA transposon', 'Caenorhabditis briggsae']
['MINISAT6_CB', 'MSAT', 'Caenorhabditis briggsae']
['MINISAT4_CB', 'MSAT', 'Caenorhabditis briggsae']


## Количество уникальных классов

In [8]:
def extract_second_column_unique(headers):
    second_values = []

    for h in headers:
        parts = h.split("\t")
        if len(parts) > 1:
            second_values.append(parts[1])

    unique_values = sorted(set(second_values))
    return unique_values


unique_second_column = extract_second_column_unique(headers)

print(f"Уникальных значений во 2-й колонке: {len(unique_second_column)}")

print(unique_second_column)

Уникальных значений во 2-й колонке: 396
['(CA)n related', '2109A repetitive sequence', '5S_DM', '5SrRNA_AN', 'AFRP1', 'ALAD', 'ALBAMH1', 'AMTAM2', 'APO1_AP', 'APO2_AP', 'ARS406', 'ARS_TA', 'AT-rich DNA repeat', 'ATREP18', 'ATREP19', 'AVIXHoI', 'AY1 repetitive sequence', 'Academ', 'AlKe1_AL', 'AlKe6_AL', 'Ambal', 'Athena', 'BDDF1', 'BEL', 'BHIKHARI_I', 'BMRP1', 'BNTAN', 'BR6_CP', 'BamHI repetitive sequence', 'BstUI repeat', 'CAM2_GG', 'CARO_CA', 'CCRP1', 'CEHAE', 'CEN1_SP', 'CENSTRIG', 'CEREP3', 'CEREP4', 'CEREP5', 'CERP1', 'CERP16', 'CERP2', 'CERP3', 'CERP4', 'CEU86951', 'CMREP', 'CPTAN', 'CR1', 'CRE', 'CRTOC1', 'CSP2034', 'CSP2090', 'CSP2111', 'CSP2112', 'C_OC', 'Caulimoviridae', 'Charlie-Galluhop', 'CoeEFV-LTR', 'Copia', 'Coprina', 'Crack', 'Crypton', 'CryptonA', 'CryptonF', 'CryptonI', 'CryptonS', 'CryptonV', 'D1100 family', 'D88I', 'DDTDD', 'DEC1_DS', 'DIRS', 'DMFTZ', 'DMFUSHI', 'DMHETRP', 'DMHMR1', 'DMHMR2', 'DNA Virus', 'DNA transposon', 'DRB_RN', 'Dada', 'Daphne', 'Dispersed rep

## Количество значений для каждого класса

In [9]:
from collections import Counter

def count_second_column(headers):
    second_values = []

    for h in headers:
        parts = h.split("\t")
        if len(parts) > 1:
            if parts[0] in remove_ids:
                continue
            second_values.append(parts[1])

    counts = Counter(second_values)
    return counts


counts = count_second_column(headers)

print(f"Всего уникальных значений: {len(counts)}")

print("\nТоп-20 самых частых:")
for name, cnt in counts.most_common(20):
    print(f"{name}: {cnt}")

Всего уникальных значений: 396

Топ-20 самых частых:
Gypsy: 32902
Copia: 11531
BEL: 7983
hAT: 7195
Mariner/Tc1: 4229
L1: 3867
ERV1: 3415
MuDR: 2630
Harbinger: 2394
DNA transposon: 2288
Helitron: 2012
ERV2: 1942
DIRS: 1607
EnSpm/CACTA: 1454
SINE2/tRNA: 1040
CR1: 1028
LTR Retrotransposon: 951
RTEX: 919
Kolobok: 903
Neptune: 900


In [11]:
rep_dict = {
    'Transposable Element': {
        'DNA transposon': {
            'Mariner/Tc1': {},
            'hAT': {},
            'MuDR': {},
            'EnSpm/CACTA': {},
            'piggyBac': {},
            'P': {},
            'Merlin': {},
            'Harbinger': {},
            'Transib': {},
            'Novosib': {},
            'Helitron': {},
            'Polinton': {},
            'Kolobok': {},
            'ISL2EU': {},
            'Crypton': {
                'CryptonA': {},
                'CryptonF': {},
                'CryptonI': {},
                'CryptonS': {},
                'CryptonV': {}
            },
            'Sola': {
                'Sola1': {},
                'Sola2': {},
                'Sola3': {}
            },
            'Zator': {},
            'Ginger1': {},
            'Ginger2/TDD': {},
            'Academ': {},
            'Zisupton': {},
            'IS3EU': {},
            'Dada': {},
            'IS481EU': {},
            'Replitron': {}
        },
        'LTR Retrotransposon': {
            'Gypsy': {},
            'Copia': {},
            'BEL': {},
            'DIRS': {},
            'Troyka': {},
            'Lodin': {}
        },
        'Endogenous Retrovirus': {
            'ERV1': {},
            'ERV2': {},
            'ERV3': {},
            'Lentivirus': {},
            'ERV4': {},
            'Lokiretrovirus': {},
            'Spumaretrovirus': {}
        },
        'Non-LTR Retrotransposon': {
            'SINE': {
                'SINE1/7SL': {},
                'SINE2/tRNA': {},
                'SINE3/5S': {},
                'SINE4': {},
                'SINEU/snRNA': {}
            },
            'CRE': {},
            'NeSL': {},
            'R4': {},
            'R2': {},
            'L1': {},
            'RTE': {},
            'I': {},
            'Jockey': {},
            'CR1': {},
            'Rex1': {},
            'RandI': {},
            'Penelope': {
                'Penelope/Poseidon': {},
                'Neptune': {},
                'Nematis': {},
                'Athena': {},
                'Coprina': {},
                'Hydra': {},
                'Naiad/Chlamys': {}
            },
            'Tx1': {},
            'RTEX': {},
            'Crack': {},
            'Nimb': {},
            'Proto1': {},
            'Proto2': {},
            'RTETP': {},
            'Hero': {},
            'L2': {},
            'Tad1': {},
            'Loa': {},
            'Ingi': {},
            'Outcast': {},
            'R1': {},
            'Daphne': {},
            'L2A': {},
            'L2B': {},
            'Ambal': {},
            'Vingi': {},
            'Kiri': {}
        }
    },
    'Simple Repeat': {
        'Satellite': {
            'SAT': {},
            'MSAT': {}
        }
    },
    'Multicopy gene': {
        'rRNA': {},
        'tRNA': {},
        'snRNA': {}
    },
    'Integrated Virus': {
        'DNA Virus': {},
        'Caulimoviridae': {}
    }
}

In [12]:
transposable_elements = rep_dict['Transposable Element']

## Общее число классов в RepBase

In [13]:
def count_nodes(tree: dict) -> int:
    total = 0
    for key, subtree in tree.items():
        total += 1              # считаем сам текущий узел
        total += count_nodes(subtree)  # считаем всех потомков
    return total
count_nodes(transposable_elements)

95

## Создаем словарь из элементов

In [14]:
def annotate_counts(tree: dict, filtered_counts: dict) -> dict:
    """
    Возвращает новое дерево, где для каждого узла есть:
    - self_count: число последовательностей именно этого типа
    - subtree_count: число последовательностей этого типа + всех потомков
    - children: дочерние узлы в таком же формате
    """
    result = {}

    for name, subtree in tree.items():
        children_annotated = annotate_counts(subtree, filtered_counts)
        self_count = filtered_counts.get(name, 0)
        subtree_count = self_count + sum(
            child_data["subtree_count"] for child_data in children_annotated.values()
        )

        result[name] = {
            "self_count": self_count,
            "subtree_count": subtree_count,
            "children": children_annotated,
        }

    return result

In [15]:
annotated_taxonomy = annotate_counts(transposable_elements, counts)

In [16]:
annotated_taxonomy

{'DNA transposon': {'self_count': 2288,
  'subtree_count': 26947,
  'children': {'Mariner/Tc1': {'self_count': 4229,
    'subtree_count': 4229,
    'children': {}},
   'hAT': {'self_count': 7195, 'subtree_count': 7195, 'children': {}},
   'MuDR': {'self_count': 2630, 'subtree_count': 2630, 'children': {}},
   'EnSpm/CACTA': {'self_count': 1454, 'subtree_count': 1454, 'children': {}},
   'piggyBac': {'self_count': 663, 'subtree_count': 663, 'children': {}},
   'P': {'self_count': 340, 'subtree_count': 340, 'children': {}},
   'Merlin': {'self_count': 179, 'subtree_count': 179, 'children': {}},
   'Harbinger': {'self_count': 2394, 'subtree_count': 2394, 'children': {}},
   'Transib': {'self_count': 252, 'subtree_count': 252, 'children': {}},
   'Novosib': {'self_count': 9, 'subtree_count': 9, 'children': {}},
   'Helitron': {'self_count': 2012, 'subtree_count': 2012, 'children': {}},
   'Polinton': {'self_count': 263, 'subtree_count': 263, 'children': {}},
   'Kolobok': {'self_count': 90

## Строим граф классов

In [ ]:
# !pip install graphviz

In [ ]:
from graphviz import Digraph


def build_graphviz_tree(annotated_tree: dict, output_name: str = "taxonomy_tree"):
    dot = Digraph(comment="Taxonomy tree")
    dot.attr(rankdir="TB")
    dot.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

    node_counter = 0

    def add_nodes(subtree: dict, parent_id: str | None = None):
        nonlocal node_counter

        for name, data in subtree.items():
            node_id = f"node_{node_counter}"
            node_counter += 1

            label = f"{name}\n{data['subtree_count']}"
            dot.node(node_id, label)

            if parent_id is not None:
                dot.edge(parent_id, node_id)

            add_nodes(data["children"], node_id)

    add_nodes(annotated_tree)

    dot.render(output_name, format="pdf", cleanup=True)
    return dot

In [ ]:
build_graphviz_tree(annotated_taxonomy, output_name=f"{config.DIR_FIGURES}/transposon_taxonomy_tree_03")

In [ ]:
annotated_taxonomy

## Оставляем только классы, для которых последовательностей больше 50

In [17]:
def prune_tree(tree: dict, threshold: int = 50) -> dict:
    """
    Удаляет узлы, у которых subtree_count < threshold
    """
    pruned = {}

    for name, data in tree.items():
        # сначала обрабатываем детей
        children_pruned = prune_tree(data["children"], threshold)

        # проверяем текущий узел
        if data["subtree_count"] >= threshold:
            pruned[name] = {
                "self_count": data["self_count"],
                "subtree_count": data["subtree_count"],
                "children": children_pruned,
            }

    return pruned

In [18]:
pruned_taxonomy = prune_tree(annotated_taxonomy, threshold=50)

In [ ]:
build_graphviz_tree(pruned_taxonomy, output_name=f"{config.DIR_FIGURES}/transposon_taxonomy_tree_04")

In [ ]:
def get_all_names(tree: dict) -> list:
    return [
        name
        for name, data in tree.items()
        for name in ([name] + get_all_names(data["children"]))
    ]

In [ ]:
names = get_all_names(pruned_taxonomy)
print(names)

In [ ]:
pruned_taxonomy

## Сохраняем данные в директорию в виде иерархии файлов и директорий

In [26]:
from pathlib import Path
from typing import Dict, List, Tuple, Optional

input_dir = config.DIR_REPBASE_RAW
output_dir = config.DIR_REPBASE_HIER

def is_input_fasta(filename: str) -> bool:
    return filename.endswith("rep.ref")



def safe_dir_name(name: str) -> str:
    return name.replace("/", "／").replace("\\", "＼")


def parse_fasta(path: str):
    """
    Итератор по fasta-записям.
    Возвращает пары: (header, sequence_text)
    header - строка с '>'
    sequence_text - строка(и) последовательности как в файле
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        header = None
        seq_lines = []

        for line in f:
            if line.startswith(">"):
                if header is not None:
                    yield header.rstrip("\n"), "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line)

        if header is not None:
            yield header.rstrip("\n"), "".join(seq_lines)


def extract_classification(header: str) -> Optional[str]:
    """
    Берёт второе поле заголовка после split по табу.
    Например:
    >id123\tDNA transposon\t....
    -> вернёт 'DNA transposon'
    """
    parts = header[1:].rstrip("\n").split("\t")
    if len(parts) < 2:
        return None
    if parts[0] in remove_ids:
        print("was", parts[0])
        return None
    return parts[1].strip()


def build_class_maps(tree: Dict, output_dir: str) -> Tuple[Dict[str, List[str]], Dict[Tuple[str, ...], Path]]:
    class_to_ancestors = {}
    path_to_output_dir = {}

    def walk(subtree: Dict, ancestors: List[str]):
        for class_name, node in subtree.items():
            current_path = ancestors + [class_name]
            class_to_ancestors[class_name] = current_path
            path_to_output_dir[tuple(current_path)] = Path(
                output_dir, *[safe_dir_name(x) for x in current_path]
            )

            children = node.get("children", {})
            if children:
                walk(children, current_path)

    walk(tree, [])
    return class_to_ancestors, path_to_output_dir


def create_output_hierarchy(path_to_output_dir: Dict[Tuple[str, ...], Path]) -> None:
    """
    Создаёт все папки и пустые seqs.fasta внутри них.
    """
    for dir_path in path_to_output_dir.values():
        dir_path.mkdir(parents=True, exist_ok=True)
        seq_file = dir_path / "seqs.fasta"
        if not seq_file.exists():
            seq_file.touch()


def write_record(out_handle, header: str, sequence: str) -> None:
    out_handle.write(header + "\n")
    out_handle.write(sequence)
    if not sequence.endswith("\n"):
        out_handle.write("\n")


def sort_sequences_by_hierarchy(input_dir: str, output_dir: str, hierarchy: Dict) -> None:
    class_to_ancestors, path_to_output_dir = build_class_maps(hierarchy, output_dir)
    create_output_hierarchy(path_to_output_dir)

    open_handles = {}
    for class_path, dir_path in path_to_output_dir.items():
        seq_file = dir_path / "seqs.fasta"
        open_handles[class_path] = open(seq_file, "a", encoding="utf-8")

    not_found_classes = set()
    skipped_headers = 0
    total_records = 0
    written_records = 0

    try:
        for dirpath, _, filenames in os.walk(input_dir):
            for filename in filenames:
                if not is_input_fasta(filename):
                    continue

                file_path = os.path.join(dirpath, filename)

                for header, sequence in parse_fasta(file_path):
                    total_records += 1

                    classification = extract_classification(header)
                    if not classification:
                        skipped_headers += 1
                        continue

                    # 1) обычный случай: классификация совпадает с именем узла
                    if classification in class_to_ancestors:
                        class_path = class_to_ancestors[classification]

                    else:
                        # 2) запасной вариант:
                        # если в header лежит полный путь, например:
                        # "DNA transposon;Mariner/Tc1"
                        # или "DNA transposon > Mariner/Tc1"
                        class_path = try_parse_full_path(classification, path_to_output_dir)

                    if class_path is None:
                        not_found_classes.add(classification)
                        continue

                    # записываем последовательность во все папки по пути от корня до листа
                    for i in range(1, len(class_path) + 1):
                        partial_path = tuple(class_path[:i])
                        write_record(open_handles[partial_path], header, sequence)

                    written_records += 1

    finally:
        for handle in open_handles.values():
            handle.close()

    print(f"Всего записей FASTA: {total_records}")
    print(f"Успешно распределено: {written_records}")
    print(f"Пропущено из-за плохого заголовка: {skipped_headers}")
    print(f"Не найдено в иерархии классов: {len(not_found_classes)}")

    if not_found_classes:
        print("\nКлассы, которых не оказалось в иерархии:")
        for cls in sorted(not_found_classes):
            print(cls)


def try_parse_full_path(classification: str, path_to_output_dir: Dict[Tuple[str, ...], Path]) -> Optional[List[str]]:
    """
    Пытается распарсить classification как полный путь по иерархии.
    Поддерживаем несколько разделителей.
    """
    candidate_delimiters = [";", " > ", "->", "|"]

    for delim in candidate_delimiters:
        if delim in classification:
            parts = [x.strip() for x in classification.split(delim) if x.strip()]
            if tuple(parts) in path_to_output_dir:
                return parts

    return None


sort_sequences_by_hierarchy(input_dir, output_dir, pruned_taxonomy)

was HAT1_CB
was DNA8-1_CB
was HELITRON9D_CB
was MirageN1b_CB
was DNA8-3_CB
was GYPSY5-I_CB
was DNA8-11_CB
was MARINER47_CB
was piggyBac7_CB
was GYPSY6-I_CB
was MARINER52_CB
was Copia-1_CI-LTR
was RhiBie-1.179
was PilTep-1.244
was DauMad-1.149
was AotNan-6.1565
was LTR8E_Mim
was NycCou-5.300
was PygNem-1.99
was DauMad-1.23
was MicMur-1.283
was MER6A
was PapAnu-1.152
was PapAnu-1.342_int
was Alu2_TS
was NycCou-1.25
was ProCoq-1.121
was OtoGar-6.322
was DauMad-6.3044
was ColAng-4.114
was CalDon-5.686
was ColAng-1.167_LTR
was EulFul-1.200
was PonAbe-1.32
was RhiRox-1.149
was MirCoq-1.224
was MirCoq-5.704
was PygNem-1.248_LTR
was PitPit-2.25
was LTR20B_Mim
was LemCat-1113
was CebCap-4.127
was IndInd-4.98
was AotNan-1.189
was SagImp-1.31
was MacERVK1_LTR1b
was SagImp-5.1072
was LTR5_RhiRox
was LemCat-1.236
was PTERV2b
was EulFul-5.499
was NomLeu-2.42
was LemCat-5.724
was IndInd-1.204
was ManLeu-1.64
was EryPat-1.273
was Buster_MacFas
was PilTep-4.374
was CheMed-1.1
was RhiBie-1.23
was PonAbe

## Сохраняем иерархию в отдельный файл с ID последовательностей

In [27]:
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional


def extract_ids_from_fasta(seq_file: Path) -> List[str]:
    """
    Читает seqs.fasta и вытаскивает id из заголовков.
    id = первое поле после split по табу.
    Например:
    >id123\tDNA transposon\t...
    -> id123
    """
    ids = []

    if not seq_file.exists():
        return ids

    with open(seq_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith(">"):
                continue

            parts = line[1:].rstrip("\n").split("\t")
            if not parts:
                continue

            seq_id = parts[0].strip()
            if seq_id in remove_ids:
                print("was", seq_id)
                continue
            if seq_id:
                ids.append(seq_id)

    return ids


def unique_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    result = []

    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)

    return result


def build_sequences_tree_from_hierarchy(
    hierarchy: Dict,
    output_dir: str
) -> Dict:
    """
    Строит структуру:
    {
      class_name: {
        "sequences": [...],
        "subs": {
          subclass_name: {...}
        }
      }
    }

    Использует уже созданную директорию output_dir и seqs.fasta внутри каждой папки.
    """

    def walk(subtree: Dict, ancestors: List[str]) -> Dict:
        result = {}

        for class_name, node in subtree.items():
            current_path = ancestors + [class_name]
            dir_path = Path(output_dir, *[safe_dir_name(x) for x in current_path])
            seq_file = dir_path / "seqs.fasta"

            sequences = unique_preserve_order(extract_ids_from_fasta(seq_file))

            children = node.get("children", {})
            subs = walk(children, current_path) if children else {}

            result[class_name] = {
                "sequences": sequences,
                "subs": subs
            }

        return result

    return walk(hierarchy, [])


def save_sequences_tree_json(
    hierarchy: Dict,
    output_dir: str,
    json_path: str
) -> None:
    tree = build_sequences_tree_from_hierarchy(hierarchy, output_dir)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(tree, f, ensure_ascii=False, indent=2)

    print(f"JSON сохранён в: {json_path}")

In [29]:
json_output_path = f"{config.DIR_REPBASE_PROCESSED}/hierarchy_sequences.json"
save_sequences_tree_json(pruned_taxonomy, output_dir, str(json_output_path))

JSON сохранён в: /Users/nad/hse/semester08/mobiraph/data/n13_repbase_processed/hierarchy_sequences.json


In [ ]:
import os


def write_all_hierarchy_sequences_to_one_fasta(
    input_dir: str,
    output_fasta: str,
    hierarchy: dict
) -> None:
    """
    Собирает все последовательности, относящиеся к hierarchy,
    в один общий fasta-файл.

    Последовательность записывается РОВНО ОДИН раз.
    """

    class_to_ancestors, path_to_output_dir = build_class_maps(hierarchy)

    total_records = 0
    written_records = 0
    skipped_headers = 0
    not_found_classes = set()

    with open(output_fasta, "w", encoding="utf-8") as out_f:
        for dirpath, _, filenames in os.walk(input_dir):
            for filename in filenames:
                if not is_input_fasta(filename):
                    continue

                file_path = os.path.join(dirpath, filename)

                for header, sequence in parse_fasta(file_path):
                    parts = header[1:].strip().split("\t")
                    if parts[0] in remove_ids:
                            continue
                    total_records += 1

                    classification = extract_classification(header)
                    if not classification:
                        skipped_headers += 1
                        continue

                    # обычный случай: classification = имя узла
                    if classification in class_to_ancestors:
                        write_record(out_f, header, sequence)
                        written_records += 1
                        continue

                    # запасной вариант: classification задан как полный путь
                    class_path = try_parse_full_path(classification, path_to_output_dir)
                    if class_path is not None:
                        write_record(out_f, header, sequence)
                        written_records += 1
                    else:
                        not_found_classes.add(classification)

    print(f"Всего записей FASTA: {total_records}")
    print(f"Записано в общий файл: {written_records}")
    print(f"Пропущено из-за плохого заголовка: {skipped_headers}")
    print(f"Не найдено в иерархии классов: {len(not_found_classes)}")

    if not_found_classes:
        print("\nКлассы, которых не оказалось в hierarchy:")
        for cls in sorted(not_found_classes):
            print(cls)

## Создадим файл с последовательностями после фильтрации

In [ ]:
if __name__ == "__main__":
    sort_sequences_by_hierarchy(input_dir, output_dir, pruned_taxonomy)

    write_all_hierarchy_sequences_to_one_fasta(
        input_dir=input_dir,
        output_fasta=f"{config.DIR_REPBASE_PROCESSED}/all_sequences_filtered_01.fasta", # после blast
        hierarchy=pruned_taxonomy
    )

## Создадим файл со всеми последовательностями RepBase

In [ ]:
def merge_all_fasta(input_dir, output_file):
    with open(output_file, "w", encoding="utf-8") as out:
        for dirpath, _, filenames in os.walk(input_dir):
            for filename in filenames:
                if not filename.endswith("rep.ref"):
                    continue

                file_path = os.path.join(dirpath, filename)

                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    for line in f:
                        out.write(line)

In [ ]:
merge_all_fasta(config.DIR_REPBASE_RAW, f"{config.DIR_REPBASE_PREPROCESSED}/all_sequences.fasta")

## Сохраняем связь между индексами последовательностей, классами, и организмами

In [ ]:
import os


def parse_fasta_metadata(input_dir):
    """
    Возвращает словарь:
    {
        seq_id: {
            "type": класс,
            "animal": организм
        }
    }
    """

    result = {}

    for dirpath, _, filenames in os.walk(input_dir):
        for filename in filenames:
            if not filename.endswith("rep.ref"):
                continue

            file_path = os.path.join(dirpath, filename)

            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    if line.startswith(">"):
                        parts = line[1:].strip().split("\t")


                        if len(parts) < 3:
                            print("Пропущено", line)
                            continue  # пропускаем кривые заголовки


                        seq_id = parts[0]
                        seq_type = parts[1]
                        animal = parts[2]

                        result[seq_id] = {
                            "type": seq_type,
                            "animal": animal
                        }

    return result

In [ ]:
metadata = parse_fasta_metadata(config.DIR_REPBASE_RAW)

print(metadata["DauMad-1.149"])

In [ ]:
import json


def save_metadata_to_json(metadata, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

In [ ]:
save_metadata_to_json(metadata, f"{config.DIR_REPBASE_PROCESSED}/metadata.json")